 # Dialogue-level BLEU on symbolic annotation tokens (MultiWOZ vs LLM annotations)

 We compare LLM-generated MultiWOZ-style annotations against the original MultiWOZ labels.
 Because the data is symbolic (dialog acts + slot/value pairs), we:
 1) linearize each turn into a canonical representation
 2) tokenize into symbols (acts and slot=value)
 3) concatenate turns per dialogue, inserting <TURN> boundaries
 4) compute corpus BLEU-1 and BLEU-4 using SacreBLEU

In [37]:
import json
import re
from collections import defaultdict
from json import JSONDecodeError
from sacrebleu.metrics import BLEU

In [38]:
from pathlib import Path

from collections import defaultdict

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RES_DIR = Path("../results")
RES_DIR.mkdir(parents=True, exist_ok=True)

DIALOGS_PATH = RAW_DIR / "multiwoz_original.json"
ACTS_PATH = RAW_DIR / "dialogue_acts.json"
ANN1_PATH = PROC_DIR / "ann1.txt"
COMMON_IDS_PATH = PROC_DIR / "common_dialog_ids.json"

print("DIALOGS:", DIALOGS_PATH.resolve(), DIALOGS_PATH.exists())
print("ACTS:   ", ACTS_PATH.resolve(), ACTS_PATH.exists())
print("ANN1:   ", ANN1_PATH.resolve(), ANN1_PATH.exists())
print("COMMON: ", COMMON_IDS_PATH.resolve(), COMMON_IDS_PATH.exists())



DIALOGS: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\multiwoz_original.json True
ACTS:    C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\dialogue_acts.json True
ANN1:    C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\ann1.txt True
COMMON:  C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\processed\common_dialog_ids.json True


In [39]:
with COMMON_IDS_PATH.open("r", encoding="utf-8") as f:
    common_ids = set(json.load(f))

with DIALOGS_PATH.open("r", encoding="utf-8") as f:
    dialogs = json.load(f)

with ACTS_PATH.open("r", encoding="utf-8") as f:
    orig_acts = json.load(f)

print("Common IDs:", len(common_ids))
print("Dialogs loaded:", len(dialogs))
print("Dialogue acts loaded:", len(orig_acts))


Common IDs: 10438
Dialogs loaded: 10438
Dialogue acts loaded: 10438


In [40]:
def parse_annotation_txt(path):
    """Parse the custom TXT annotation export into a MultiWOZ-like dict.

    Each line:
        <utterance>\t<json1>[\t<json2>\t<json3>...]

    Returns:
        dict[dialog_id] -> {"log_by_turn": dict[int, turn_obj]}
    """
    dialogs = defaultdict(lambda: {"log_by_turn": {}})

    stats = {
        "lines_total": 0,
        "lines_no_json": 0,
        "lines_no_base": 0,
        "lines_no_da": 0,
        "lines_parsed": 0,
        "turn_overwrites": 0,
    }

    with open(path, encoding="utf-8") as f:
        for lineno, raw_line in enumerate(f, start=1):
            stats["lines_total"] += 1

            line = raw_line.rstrip("\n")
            if not line.strip():
                continue

            parts = line.split("\t")
            if len(parts) < 2:
                stats["lines_no_json"] += 1
                continue

            utterance = parts[0]

            json_strs = []
            for p in parts[1:]:
                p = p.strip()
                if not p:
                    continue
                if "{" in p and "}" in p:
                    start = p.index("{")
                    end = p.rfind("}") + 1
                    json_strs.append(p[start:end])

            if not json_strs:
                stats["lines_no_json"] += 1
                continue

            objs = []
            for js in json_strs:
                try:
                    objs.append(json.loads(js))
                except JSONDecodeError as e:
                    print(f"[ERROR] JSON parse failed on line {lineno}: {e}")
                    print("  Chunk:", repr(js))
                    raise

            base = None
            for o in objs:
                if "_dialog_index" in o and "_turn_index" in o:
                    base = o
                    break
            if base is None:
                stats["lines_no_base"] += 1
                continue

            dialog_id = base["_dialog_index"]
            t_idx = int(base["_turn_index"])

            speaker = next((o.get("speaker") for o in objs if "speaker" in o), "unknown")

            dialog_act = {}
            for obj in objs:
                act_name = obj.get("dialog_act")
                if not act_name:
                    continue
                slots = obj.get("slots", {}) or {}

                # IMPORTANT: keep raw values (do NOT str() here) so strict/normalized stay meaningful
                slot_list = [[k, v] for k, v in slots.items()]

                dialog_act.setdefault(act_name, []).extend(slot_list)

            if not dialog_act:
                stats["lines_no_da"] += 1
                continue

            turn_obj = {"text": utterance, "dialog_act": dialog_act, "speaker": speaker}

            if t_idx in dialogs[dialog_id]["log_by_turn"]:
                stats["turn_overwrites"] += 1

            dialogs[dialog_id]["log_by_turn"][t_idx] = turn_obj
            stats["lines_parsed"] += 1

    print("parse_annotation_txt stats:", stats)
    return dict(dialogs)

cand_data = parse_annotation_txt(ANN1_PATH)
print("Loaded candidate dialogs:", len(cand_data))


parse_annotation_txt stats: {'lines_total': 144441, 'lines_no_json': 1404, 'lines_no_base': 0, 'lines_no_da': 34, 'lines_parsed': 143000, 'turn_overwrites': 0}
Loaded candidate dialogs: 10438


In [41]:
def normalize_slot_value(slot, slot_val, mode="strict"):
    """Normalize a slot value according to mode.

    :param slot: normalized slot name (lowercase)
    :param slot_val: raw slot value (any type)
    :param mode: "strict" | "normalized" | "lenient"
    :return: normalized string (possibly empty)
    """
    if mode == "strict":
        return "None" if slot_val is None else str(slot_val).strip()

    sval = "" if slot_val is None else str(slot_val).strip().lower()

    if sval in {"none", "null", "nan", "n/a", "?"}:
        sval = ""

    if slot in {"wifi", "internet", "parking"}:
        truthy = {"true", "yes", "y", "1"}
        falsy = {"false", "no", "n", "0"}
        if mode == "lenient":
            truthy |= {"free", "available"}

        if sval in truthy:
            sval = "yes"
        elif sval in falsy:
            sval = "no"

    return sval

In [42]:
def linearize_turn(turn_obj, mode="strict"):
    """Linearize one turn into a canonical string: 'act1,act2(slot=a,slot=b)'.

    - acts kept in insertion order
    - all slot/value pairs collected across acts
    - slot pairs sorted alphabetically (preserve duplicates)
    - empty values kept as 'slot='

    :param turn_obj: dict with "dialog_act"
    :param mode: "strict" | "normalized" | "lenient"
    :return: canonical string
    """
    da = turn_obj.get("dialog_act", {}) or {}
    act_names = list(da.keys())  # insertion order

    if mode not in {"strict", "normalized", "lenient"}:
        raise ValueError(f"Unknown mode: {mode}")

    slot_pairs = []
    for act_name in act_names:
        for slot_name, slot_val in da.get(act_name, []) or []:
            slot = str(slot_name).strip().lower()
            sval = normalize_slot_value(slot, slot_val, mode=mode)
            slot_pairs.append((slot, sval))

    acts_part = ",".join(str(a).strip().lower() for a in act_names)

    slot_pairs.sort(key=lambda x: (x[0], x[1]))
    slots_str = ",".join(f"{k}={v}" for k, v in slot_pairs) if slot_pairs else ""

    return f"{acts_part}({slots_str})"

In [43]:
def bleu_tokenize(linear_str):
    """Tokenize a linearized turn string into BLEU tokens (acts + slot=value)."""
    s = linear_str.strip()

    if "(" in s and ")" in s:
        acts_part, slots_part = s.split("(", 1)
        slots_part = slots_part.rsplit(")", 1)[0]
    else:
        acts_part, slots_part = s, ""

    act_tokens = [t.strip() for t in acts_part.split(",") if t.strip()]
    slot_tokens = [t.strip() for t in slots_part.split(",") if t.strip()]

    tokens = [re.sub(r"\s+", "", t) for t in (act_tokens + slot_tokens)]
    return [t for t in tokens if t]

In [44]:
def build_dialog_token_sequences(data, mode="strict", add_turn_boundary=True):
    """Build dict[dialog_id] -> flattened token sequence for BLEU.

    Supports:
      - original MultiWOZ: {"log": [turn0, turn1, ...]}
      - candidate: {"log_by_turn": {idx: turn_obj, ...}}
    """
    out = {}

    for d_id, dialog in data.items():
        tokens = []

        if dialog.get("log_by_turn"):
            for t_idx in sorted(dialog["log_by_turn"].keys()):
                turn = dialog["log_by_turn"][t_idx]
                linear = linearize_turn(turn, mode=mode)
                tokens.extend(bleu_tokenize(linear))
                if add_turn_boundary:
                    tokens.append("<TURN>")
        else:
            log = dialog.get("log", []) or []
            for turn in log:
                linear = linearize_turn(turn, mode=mode)
                tokens.extend(bleu_tokenize(linear))
                if add_turn_boundary:
                    tokens.append("<TURN>")

        out[d_id] = tokens

    return out

In [45]:
bleu1_metric = BLEU(max_ngram_order=1, effective_order=True)
bleu4_metric = BLEU(max_ngram_order=4, effective_order=True)

def compute_bleu_scores(data_ref, data_hyp, mode="strict"):
    """Compute corpus BLEU-1 and BLEU-4 for a given mode."""
    ref_dialogs = build_dialog_token_sequences(data_ref, mode=mode, add_turn_boundary=True)
    hyp_dialogs = build_dialog_token_sequences(data_hyp, mode=mode, add_turn_boundary=True)

    common_ids = sorted(set(ref_dialogs) & set(hyp_dialogs))

    refs, hyps = [], []
    for d in common_ids:
        r = " ".join(ref_dialogs[d]).strip()
        h = " ".join(hyp_dialogs[d]).strip()
        if not r or not h:
            continue
        refs.append(r)
        hyps.append(h)

    return {
        "mode": mode,
        "dialogs_used": len(refs),
        "bleu1": bleu1_metric.corpus_score(hyps, [refs]).score,
        "bleu4": bleu4_metric.corpus_score(hyps, [refs]).score,
        "turn_boundary": True,
    }

In [46]:
for m in ["strict", "normalized", "lenient"]:
    print(m, compute_bleu_scores(dialogs, cand_data, mode=m))

strict {'mode': 'strict', 'dialogs_used': 10438, 'bleu1': 63.806533248061, 'bleu4': 36.6620245783474, 'turn_boundary': True}
normalized {'mode': 'normalized', 'dialogs_used': 10438, 'bleu1': 70.17048177913554, 'bleu4': 42.21928198648245, 'turn_boundary': True}
lenient {'mode': 'lenient', 'dialogs_used': 10438, 'bleu1': 70.32382307612647, 'bleu4': 42.51524818203763, 'turn_boundary': True}


In [47]:
# Sanity check: show one dialog token stream head
ref_strict = build_dialog_token_sequences(dialogs, mode="strict", add_turn_boundary=True)
hyp_strict = build_dialog_token_sequences(cand_data, mode="strict", add_turn_boundary=True)

common_ids = sorted(set(ref_strict) & set(hyp_strict))
sample_id = common_ids[0]

print("Sample dialog:", sample_id)
print("REF (first 40):", ref_strict[sample_id][:40])
print("HYP (first 40):", hyp_strict[sample_id][:40])

Sample dialog: MUL0001.json
REF (first 40): ['restaurant-inform', 'food=indian', '<TURN>', 'restaurant-request', 'restaurant-inform', 'area=?', 'choice=many', 'food=Indian', 'price=thatpricerange', '<TURN>', 'restaurant-inform', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 'booking-inform', 'restaurant-recommend', 'area=center', 'food=Indian', 'name=SaffronBrasserie', 'none=none', 'price=expensive', '<TURN>', 'restaurant-inform', 'day=saturday', 'people=6', 'time=19:30', '<TURN>', 'booking-book', 'ref=RF00JUFQ', '<TURN>', 'hotel-inform', 'internet=no', 'stars=3', 'type=hotel', '<TURN>', 'booking-inform', 'hotel-inform', 'area=north', 'internet=none']
HYP (first 40): ['restaurant-request', 'food=indian', '<TURN>', 'restaurant-inform', 'restaurant-request', 'food=indian', 'price=', '<TURN>', 'restaurant-request', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 'restaurant-recommend', 'booking-offerbook', 'area=centre', 'food=indian', 'price=expensive', '<TURN>', 

In [48]:
# --- BLEU-only results dump (matches what you print above) ---
import json

bleu_results = {
    "strict": compute_bleu_scores(dialogs, cand_data, mode="strict"),
    "normalized": compute_bleu_scores(dialogs, cand_data, mode="normalized"),
    "lenient": compute_bleu_scores(dialogs, cand_data, mode="lenient"),
}

OUT_PATH = RES_DIR / "bleu_ann1_vs_original.json"
with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(bleu_results, f, indent=2)

print("Saved:", OUT_PATH.resolve())
print(json.dumps(bleu_results, indent=2))



Saved: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\results\bleu_ann1_vs_original.json
{
  "strict": {
    "mode": "strict",
    "dialogs_used": 10438,
    "bleu1": 63.806533248061,
    "bleu4": 36.6620245783474,
    "turn_boundary": true
  },
  "normalized": {
    "mode": "normalized",
    "dialogs_used": 10438,
    "bleu1": 70.17048177913554,
    "bleu4": 42.21928198648245,
    "turn_boundary": true
  },
  "lenient": {
    "mode": "lenient",
    "dialogs_used": 10438,
    "bleu1": 70.32382307612647,
    "bleu4": 42.51524818203763,
    "turn_boundary": true
  }
}
